In [1]:
# 06_weight_sensitivity.ipynb
# Sensitivity of the optimal friction f* to +/-50% perturbations of each objective weight,
# and comparison with the closed form of Corollary 4.5 (which ignores integer staffing).

In [2]:
# ---- shared model block (identical in 05 and 06) ----
import itertools, json
import numpy as np, pandas as pd
df = pd.read_csv("../data/tasks.csv")
TIDS = df["task_id"].tolist()
tau = df["pre_ai_hours"].to_numpy(float)          # human task time; AI draft replaces it: g_i = tau_i
v = df["verification_hours"].to_numpy(float)       # time to verify a full AI draft
sev = df["error_severity"].to_numpy(float)         # 1-5 rating
e = sev / 5.0                                      # error cost of a fully delegated task (hour-equivalents)
cap = np.where(df["accountability_constraint"] == 1, 0.5, 1.0)   # accountability caps
LAM, K = 0.30, 0.6                                 # illustrative arrival rate (cases/h), friction sensitivity
W_BASE = dict(c_H=0.4, w_h=0.25, w_e=5.0)          # w_e * e_i = severity
LEVELS = (0.0, 0.25, 0.5, 0.75, 1.0)
X = np.array(list(itertools.product(*[[l for l in LEVELS if l <= cap[i]] for i in range(len(TIDS))])))
H = ((1 - X) * tau).sum(1)                         # retained human time
V = (X * v).sum(1)                                 # verification of delegated output
P = (e * X**2).sum(1)                              # convex error exposure, phi(x) = x^2
FS = np.round(np.arange(0, 12.0001, 0.05), 2)

def best_at_f(f, c_H=0.4, w_h=0.25, w_e=5.0):
    """Min over (x, c) at fixed friction f. Friction acts on v_i(f) = v_i(1 + k f) only;
    c is the smallest integer with c > lam E[S]; escaped-error cost falls as 1/(1+f)."""
    ES = H + V * (1 + K * f)
    c = np.floor(LAM * ES) + 1
    Z = c_H * c + w_h * ES + w_e * P / (1 + f)
    j = int(Z.argmin())
    return float(Z[j]), int(c[j]), X[j], float(ES[j])

def solve(**w):
    env = [best_at_f(f, **w) for f in FS]
    j = int(np.argmin([r[0] for r in env]))
    return float(FS[j]), env[j], env


In [3]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi":600,"font.size":11,"axes.edgecolor":"0.2","axes.linewidth":0.8,"grid.color":"0.85"})
def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"../results/figures/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
cases = {"baseline": {}, "staffing +50%": dict(c_H=0.6), "staffing -50%": dict(c_H=0.2),
         "error +50%": dict(w_e=7.5), "error -50%": dict(w_e=2.5),
         "hours +50%": dict(w_h=0.375), "hours -50%": dict(w_h=0.125)}
res = {}
for name, d in cases.items():
    w = dict(W_BASE); w.update(d)
    f, r, _ = solve(**w); res[name] = dict(f=f, c=r[1], x=list(map(float, r[2])))
    print(f"{name:<15} f* = {f:5.2f}  c* = {r[1]}  x* = {r[2]}")
f_closed = float(np.sqrt(sev.sum()/(W_BASE["c_H"]*LAM*K)) - 1)
print(f"Closed form (Cor 4.5, continuous staffing, E_err = sum severity): f* = {f_closed:.2f} -> integer staffing binds first")
fig, ax = plt.subplots(figsize=(6.5, 4))
names = list(res); vals = [res[n]["f"] for n in names]; yp = np.arange(len(names))
ax.barh(yp, vals, color="0.45", edgecolor="black", linewidth=0.7)
for y, n in zip(yp, names):
    ax.annotate(f"{res[n]['f']:.2f}  (c*={res[n]['c']})", (res[n]["f"], y), textcoords="offset points", xytext=(4, -3), fontsize=8)
ax.axvline(res["baseline"]["f"], color="0.15", ls="--", lw=1.2, label="baseline $f^*$")
ax.set_yticks(yp); ax.set_yticklabels(names, fontsize=9); ax.invert_yaxis()
ax.set_xlabel("Optimal friction $f^*$"); ax.set_xlim(0, 13.5); ax.legend(frameon=True, edgecolor="0.5", fontsize=9, loc="upper right")
save(fig, "fig10_weight_sensitivity")
json.dump(dict(weight_sensitivity=res, f_closed=f_closed), open("../results/tables/n1_results.json", "w"), indent=2)

baseline        f* =  5.40  c* = 2  x* = [1.  1.  0.  0.  0.5 0.  0.5]
staffing +50%   f* =  5.40  c* = 2  x* = [1.  1.  0.  0.  0.5 0.  0.5]
staffing -50%   f* =  5.40  c* = 2  x* = [1.  1.  0.  0.  0.5 0.  0.5]
error +50%      f* =  8.60  c* = 3  x* = [0.75 0.5  0.   0.   0.25 0.   0.25]
error -50%      f* =  5.40  c* = 2  x* = [1.  1.  0.  0.  0.5 0.  0.5]
hours +50%      f* =  5.40  c* = 2  x* = [1.  1.  0.  0.  0.5 0.  0.5]
hours -50%      f* = 10.75  c* = 3  x* = [0.75 0.5  0.   0.   0.   0.   0.25]
Closed form (Cor 4.5, continuous staffing, E_err = sum severity): f* = 17.26 -> integer staffing binds first
